[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/practical-molecular-dynamics-course/blob/main/notebooks/11_protein_ligand_md.ipynb)

# 11 · Protein–Ligand MD: EGFR Kinase with the Drug Erlotinib

*Practical Molecular Dynamics Course · Hamza Hentabli, Ph.D. · ~75 min*

This is where MD meets drug discovery. We take the crystal structure of the **EGFR kinase domain bound to
erlotinib** (PDB 1M17; Stamos *et al.*, *J. Biol. Chem.* 2002), an approved lung-cancer drug, and build a full
simulation: prepare the protein, give the drug **GAFF2 force-field parameters with AM1-BCC charges**, solvate the
complex, minimize, and run dynamics. Then we analyse a short precomputed trajectory the way a computational
chemist checks a docking pose: is the ligand stable, does the key **hinge hydrogen bond to Met769** hold, and which
residues does the drug touch (an **interaction fingerprint**)?

**Learning objectives.** By the end of this notebook you will be able to:

- prepare a ligand from a crystal structure with RDKit (bond orders, hydrogens, protonation)
- parameterize a drug-like molecule with GAFF2/AM1-BCC (openmmforcefields) and combine it with a protein force field
- build, solvate, and minimize a protein–ligand complex in OpenMM
- analyse ligand RMSD, a key hydrogen bond, contact frequencies, and an interaction fingerprint

In [ ]:
# Setup: installs dependencies on Google Colab and locates the course data.
# Running locally? Create the environment from environment.yml once and this cell does nothing extra.
import os, sys, subprocess, shutil

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "rdkit", "py3Dmol"], check=True)
    if not os.path.isdir("practical-molecular-dynamics-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/practical-molecular-dynamics-course.git"], check=True)
    DATA_DIR = "practical-molecular-dynamics-course/data"
else:
    DATA_DIR = "../data" if os.path.isdir("../data") else "data"

# Ligand parameterization needs openmmforcefields + the OpenFF toolkit + AmberTools (antechamber/sqm), which are
# conda packages. They are in environment.yml. On Colab the notebook uses the precomputed parameter file instead
# (see the README for an optional condacolab install that lets you run this step live).
try:
    import openmmforcefields, openff.toolkit  # noqa: F401
    CAN_PARAMETERIZE = shutil.which("antechamber") is not None and shutil.which("sqm") is not None
except ImportError:
    CAN_PARAMETERIZE = False
print("Running on Colab:", IN_COLAB, "| data folder:", DATA_DIR, "| live ligand parameterization:", CAN_PARAMETERIZE)

## 1. The crystal structure and its binding site

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openmm
from openmm import app, unit
import mdtraj as md

os.makedirs("output", exist_ok=True)
xtal = md.load(f"{DATA_DIR}/structures/1M17.pdb")
lig_idx = xtal.topology.select("resname AQ4")
prot_idx = xtal.topology.select("protein")
near = md.compute_neighbors(xtal, 0.4, lig_idx, haystack_indices=prot_idx)[0]
site = sorted({xtal.topology.atom(i).residue for i in near}, key=lambda r: r.resSeq)
print(f"erlotinib: {len(lig_idx)} heavy atoms; {len(site)} residues within 4 Å:")
print(", ".join(f"{r.name}{r.resSeq}" for r in site))

hinge = [xtal.topology.select("resname AQ4 and name N2")[0], xtal.topology.select("protein and resSeq 769 and name N")[0]]
print(f"hinge hydrogen bond, erlotinib N2 ... Met769 N: {md.compute_distances(xtal, [hinge])[0, 0] * 10:.2f} Å")

Met769 sits in the **hinge** that connects the two lobes of the kinase; its backbone NH donates a hydrogen bond to
the quinazoline N2 of erlotinib (UniProt numbering: Met793). Almost every ATP-competitive kinase inhibitor makes
such a hinge contact, so it is the first thing to watch in a simulation.

## 2. Preparing the ligand with RDKit

PDB files store ligand atoms and coordinates but no bond orders or hydrogens. We take the bond orders from
erlotinib's SMILES, then add hydrogens with 3-D coordinates. Erlotinib is modelled as **neutral**: its most basic
site, the quinazoline N1, has a pKa of about 5.4, so it is mostly unprotonated at pH 7.4 (always check ligand
protonation and tautomers; tools such as Epik, Dimorphite-DL, or MolGpKa help).

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors

ERLOTINIB = "COCCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOC"
lines = [l for l in open(f"{DATA_DIR}/structures/1M17.pdb") if l.startswith("HETATM") and l[17:20] == "AQ4"]
lig = Chem.MolFromPDBBlock("".join(lines), removeHs=False)
lig = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(ERLOTINIB), lig)
ligH = Chem.AddHs(lig, addCoords=True, addResidueInfo=True)
Chem.MolToPDBFile(ligH, "output/erlotinib_H.pdb")
print("atoms with H:", ligH.GetNumAtoms(), "| formal charge:", Chem.GetFormalCharge(ligH),
      f"| MW {Descriptors.MolWt(ligH):.1f} | rotatable bonds {rdMolDescriptors.CalcNumRotatableBonds(ligH)}")
Draw.MolToImage(Chem.MolFromSmiles(ERLOTINIB), size=(420, 260))

## 3. Force-field parameters for the drug: GAFF2 + AM1-BCC

Protein force fields only know amino acids. For drug-like molecules we use a *general* force field: **GAFF2**
(AMBER), CGenFF (CHARMM), or OpenFF Sage. `openmmforcefields` assigns GAFF2 atom types with AmberTools'
antechamber, computes **AM1-BCC partial charges** (a fast semi-empirical method that mimics HF/6-31G* electrostatic
potential charges), and writes an OpenMM force-field file. Charges take about a minute for erlotinib.

In [ ]:
precomputed = f"{DATA_DIR}/ligands/erlotinib_gaff2.xml"
if CAN_PARAMETERIZE:
    from openff.toolkit import Molecule
    from openmmforcefields.generators import GAFFTemplateGenerator
    off = Molecule.from_rdkit(ligH, allow_undefined_stereo=True)
    off.name = "AQ4"
    t0 = time.time()
    gaff = GAFFTemplateGenerator(molecules=off, forcefield="gaff-2.11")
    xml = gaff.generate_residue_template(off)
    open("output/erlotinib_gaff2.xml", "w").write(xml)
    LIGAND_XML = "output/erlotinib_gaff2.xml"
    print(f"GAFF2/AM1-BCC parameters computed live in {time.time() - t0:.0f} s")
else:
    LIGAND_XML = precomputed
    print("using precomputed parameters:", LIGAND_XML)

import xml.etree.ElementTree as ET
root = ET.parse(LIGAND_XML).getroot()
atoms = pd.DataFrame([a.attrib for a in root.iter("Atom")])
atoms["charge"] = atoms["charge"].astype(float)
print("GAFF2 atom types:", atoms["type"].value_counts().to_dict())
print(f"total charge {atoms.charge.sum():+.3f}; most negative: "
      + ", ".join(f"{r['name']} {r.charge:+.2f}" for _, r in atoms.nsmallest(3, "charge").iterrows()))

Check that the total charge is an integer (0 here), that aromatic, sp, and ether atoms received sensible GAFF2
types, and that nitrogen and oxygen atoms carry negative charges. For production work, compare the charges with
those computed by a second method or with the published parameters of a related ligand.

## 4. Build the solvated complex

Protein preparation follows Notebook 06: keep residues 672–964 (the whole kinase domain), do not build the
expression tag, and trim the C-terminal tail beyond the unresolved segment 965–976. Then add the ligand, an
octahedral water box, and 0.15 M NaCl.

In [ ]:
from pdbfixer import PDBFixer

keep = [l for l in open(f"{DATA_DIR}/structures/1M17.pdb")
        if not (l.startswith(("ATOM", "HETATM", "ANISOU")) and l[17:20] != "HOH" and int(l[22:26]) > 964)]
open("output/1M17_trimmed.pdb", "w").writelines(keep)
fixer = PDBFixer(filename="output/1M17_trimmed.pdb")
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues(); fixer.missingResidues = {}
fixer.findNonstandardResidues(); fixer.replaceNonstandardResidues()
fixer.findMissingAtoms(); fixer.addMissingAtoms()

ff = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml", LIGAND_XML)
complex_ = app.Modeller(fixer.topology, fixer.positions)
complex_.addHydrogens(ff, pH=7.0)
ligand_pdb = app.PDBFile("output/erlotinib_H.pdb")
complex_.add(ligand_pdb.topology, ligand_pdb.positions)
n_solute = complex_.topology.getNumAtoms()
t0 = time.time()
complex_.addSolvent(ff, padding=1.0 * unit.nanometer, ionicStrength=0.15 * unit.molar, boxShape="octahedron")
counts = pd.Series([r.name for r in complex_.topology.residues()]).value_counts()
print(f"solute {n_solute} atoms; total {complex_.topology.getNumAtoms()} atoms "
      f"({counts['HOH']} waters, {counts.get('NA', 0)} Na+, {counts.get('CL', 0)} Cl-) in {time.time() - t0:.0f} s")

In [ ]:
system = ff.createSystem(complex_.topology, nonbondedMethod=app.PME, nonbondedCutoff=1.0 * unit.nanometer,
                         constraints=app.HBonds, hydrogenMass=1.5 * unit.amu)
system.addForce(openmm.MonteCarloBarostat(1 * unit.bar, 300 * unit.kelvin))
integ = openmm.LangevinMiddleIntegrator(300 * unit.kelvin, 1 / unit.picosecond, 0.004 * unit.picoseconds)
integ.setRandomNumberSeed(11)
sim = app.Simulation(complex_.topology, system, integ, openmm.Platform.getPlatformByName("CPU"))
sim.context.setPositions(complex_.positions)
e0 = sim.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
t0 = time.time()
sim.minimizeEnergy(maxIterations=500)
e1 = sim.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
print(f"minimization (500 iterations max): {e0:.3e} -> {e1:.3e} kJ/mol in {time.time() - t0:.0f} s")

## 5. A short live run

A few picoseconds are enough to see that everything works. (On a CPU this system runs at roughly 10 ns/day; on a
GPU, hundreds. The precomputed trajectory below used 30 ps of restrained equilibration and 50 ps of production, which took
about 15 minutes on 5 CPU cores.)

In [ ]:
LIVE_PS = 2
sim.context.setVelocitiesToTemperature(300 * unit.kelvin, 11)
t0 = time.time()
sim.step(int(LIVE_PS / 0.004))
wall = time.time() - t0
st = sim.context.getState(getPositions=True, getEnergy=True)
print(f"{LIVE_PS} ps in {wall:.0f} s ({LIVE_PS / 1000 / wall * 86400:.1f} ns/day on this machine)")
live = md.Trajectory(np.array([st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)[:n_solute]]),
                     md.Topology.from_openmm(complex_.topology).subset(range(n_solute)))
n2 = live.topology.select("resname AQ4 and name N2")[0]; met = live.topology.select("protein and resSeq 769 and name N")[0]
print(f"hinge N2 ... Met769 N after {LIVE_PS} ps: {md.compute_distances(live, [[n2, met]])[0, 0] * 10:.2f} Å")

## 6. Analysis of the precomputed 50 ps trajectory

`data/trajectories/egfr_erlotinib.dcd` stores the protein and ligand (no water) every 1 ps, already superposed on
the protein Cα atoms. The matching topology is `egfr_erlotinib_complex.pdb`. Set `USE_LIVE = True` to analyse a
trajectory you produced yourself instead (with the same atom order).

In [ ]:
USE_LIVE = False
top = f"{DATA_DIR}/trajectories/egfr_erlotinib_complex.pdb"
traj = md.load(f"{DATA_DIR}/trajectories/egfr_erlotinib.dcd", top=top)
ref = md.load(top)                                  # the prepared crystal pose (frame of reference)
t_ps = traj.time if traj.time.max() > 0 else np.arange(1, traj.n_frames + 1) * 1.0
print(traj)

ca = traj.topology.select("protein and name CA")
lig_heavy = traj.topology.select("resname AQ4 and not element H")
traj.superpose(ref, atom_indices=ca); 
rmsd_prot = md.rmsd(traj, ref, atom_indices=ca) * 10
# ligand RMSD *without* refitting on the ligand: how far did the pose move inside the pocket?
lig_rmsd = np.sqrt(((traj.xyz[:, lig_heavy] - ref.xyz[0, lig_heavy]) ** 2).sum(-1).mean(1)) * 10
core = traj.topology.select("resname AQ4 and not element H and name " + " ".join(
    ["C1", "C2", "C3", "C4", "C5", "N1", "C6", "C7", "C8", "C9", "C16", "C17", "C18", "N2", "C19", "N3", "C20", "C21", "C22"]))
core_rmsd = np.sqrt(((traj.xyz[:, core] - ref.xyz[0, core]) ** 2).sum(-1).mean(1)) * 10
hinge_d = md.compute_distances(traj, [[traj.topology.select("resname AQ4 and name N2")[0],
                                       traj.topology.select("protein and resSeq 769 and name N")[0]]])[:, 0] * 10

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(t_ps, rmsd_prot, label="protein Cα")
ax[0].plot(t_ps, lig_rmsd, label="erlotinib, all heavy atoms")
ax[0].plot(t_ps, core_rmsd, label="erlotinib, rings + acetylene")
ax[0].set(xlabel="time (ps)", ylabel="RMSD from crystal (Å)", title="Pose stability (protein-aligned)"); ax[0].legend()
ax[1].plot(t_ps, hinge_d); ax[1].axhline(3.5, color="r", ls="--", lw=0.8, label="3.5 Å H-bond cutoff")
ax[1].set(xlabel="time (ps)", ylabel="Å", title="Hinge H-bond: erlotinib N2 ... Met769 N"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"protein Cα RMSD {rmsd_prot.mean():.2f} ± {rmsd_prot.std():.2f} Å | ligand RMSD {lig_rmsd.mean():.2f} ± {lig_rmsd.std():.2f} Å "
      f"(core {core_rmsd.mean():.2f} Å) | hinge H-bond present ({'<'}3.5 Å) in {(hinge_d < 3.5).mean() * 100:.0f}% of frames, "
      f"mean {hinge_d.mean():.2f} Å")

Read the two plots together. A pose that stays within ~2 Å of the crystal while the hinge hydrogen bond holds is
behaving like a real binder on this time scale. The flexible methoxyethoxy chains point out to solvent, so the
all-atom ligand RMSD is larger than the RMSD of the rigid core. Fifty picoseconds is only a *smoke test* of the setup, not evidence of binding:
docking poses that are wrong often fall apart within a few nanoseconds, so in practice you run several replicas of
10–100 ns.

## 7. Which residues does the drug touch?

In [ ]:
prot_heavy = traj.topology.select("protein and not element H")
contact_frames = {}
for f in range(traj.n_frames):
    close = md.compute_neighbors(traj[f], 0.4, lig_heavy, haystack_indices=prot_heavy)[0]
    for r in {traj.topology.atom(i).residue for i in close}:
        contact_frames[f"{r.name}{r.resSeq}"] = contact_frames.get(f"{r.name}{r.resSeq}", 0) + 1
contacts = (pd.Series(contact_frames) / traj.n_frames).sort_values(ascending=False)
xtal_site = {f"{r.name}{r.resSeq}" for r in site}
plt.figure(figsize=(11, 3.6))
colors = ["C0" if k in xtal_site else "C1" for k in contacts.index[:25]]
contacts.iloc[:25].plot.bar(color=colors)
plt.ylabel("fraction of frames within 4 Å"); plt.title("Erlotinib contacts during MD (blue: also in the crystal site, orange: new)")
plt.tight_layout(); plt.show()
print("residues in contact in >50% of frames:", int((contacts > 0.5).sum()), "| crystal 4 Å site:", len(xtal_site))

## 8. An interaction fingerprint

An **interaction fingerprint** (IFP) encodes, frame by frame, which residue makes which type of interaction with
the ligand. It is the standard way to compare poses, docking results, and MD frames (tools such as ProLIF and PLIP
compute richer versions). We compute three simple, transparent types:

- **H-bond**: polar N/O of the protein within 3.5 Å of a polar N/O of the ligand, with a hydrogen in between
  (from MDTraj's Baker–Hubbard criterion, evaluated frame by frame);
- **hydrophobic**: carbon–carbon contact ≤ 4.0 Å between ligand and side chain;
- **any contact**: any heavy atoms ≤ 4.0 Å.

In [ ]:
from itertools import product

lig_c = traj.topology.select("resname AQ4 and element C")
prot_sc_c = traj.topology.select("protein and element C and not name C CA")
hb_rows = []
for f in range(traj.n_frames):
    for d, h, a in md.baker_hubbard(traj[f], freq=0.0, exclude_water=True, periodic=False):
        rd, ra = traj.topology.atom(d).residue, traj.topology.atom(a).residue
        if (rd.name == "AQ4") != (ra.name == "AQ4"):
            res = ra if rd.name == "AQ4" else rd
            hb_rows.append((f, f"{res.name}{res.resSeq}"))
hb = pd.DataFrame(hb_rows, columns=["frame", "residue"]).drop_duplicates()

ifp = {}
top_res = list(contacts.index[:15])
for res in top_res:
    r_atoms = [a.index for a in traj.topology.atoms if f"{a.residue.name}{a.residue.resSeq}" == res]
    r_c = np.intersect1d(r_atoms, prot_sc_c)
    pairs_c = np.array(list(product(lig_c, r_c))) if len(r_c) else None
    hyd = (md.compute_distances(traj, pairs_c).min(1) <= 0.40) if pairs_c is not None else np.zeros(traj.n_frames, bool)
    hbond = np.isin(np.arange(traj.n_frames), hb.frame[hb.residue == res])
    ifp[(res, "H-bond")] = hbond
    ifp[(res, "hydrophobic")] = hyd
ifp = pd.DataFrame(ifp, index=t_ps)
ifp = ifp.loc[:, ifp.any()]                               # keep interactions that occur at least once

fig, ax = plt.subplots(figsize=(12, 0.32 * ifp.shape[1] + 1.2))
ax.imshow(ifp.T.values, aspect="auto", cmap="Blues", interpolation="nearest",
          extent=[t_ps[0], t_ps[-1], ifp.shape[1] - 0.5, -0.5])
ax.set_yticks(range(ifp.shape[1])); ax.set_yticklabels([f"{r} · {k}" for r, k in ifp.columns])
ax.set(xlabel="time (ps)", title="Interaction fingerprint of erlotinib in EGFR (blue = present)")
plt.tight_layout(); plt.savefig("output/egfr_erlotinib_ifp.png", dpi=150); plt.show()
ifp.mean().sort_values(ascending=False).round(2).to_frame("occupancy").head(12)

The fingerprint turns a trajectory into a table you can compare across ligands, replicas, or docking poses: for
example, "does my new analogue keep the Met769 hinge H-bond and the hydrophobic contacts of erlotinib?" Combined
with longer simulations and replicas, this kind of analysis supports decisions in lead optimization.

## 9. Look at it

In [ ]:
traj[-1].save_pdb("output/egfr_erlotinib_last.pdb")
try:
    import py3Dmol
    v = py3Dmol.view(width=600, height=420)
    v.addModel(open("output/egfr_erlotinib_last.pdb").read(), "pdb")
    v.setStyle({"cartoon": {"color": "lightgrey"}})
    v.addStyle({"resn": "AQ4"}, {"stick": {"colorscheme": "greenCarbon"}})
    v.addStyle({"resi": "769"}, {"stick": {"colorscheme": "orangeCarbon"}, "cartoon": {"color": "orange"}})
    v.zoomTo({"resn": "AQ4"}); v.show()
except ImportError:
    print("py3Dmol not installed; open output/egfr_erlotinib_last.pdb in PyMOL or ChimeraX")

**Where to go from here.** This notebook covers the MD part of a structure-based project. Next steps in a real
study: several independent replicas, longer runs, clustering of ligand poses, end-point free energies (MM/GBSA),
and alchemical relative binding free energies (FEP; e.g. OpenFE or perses on top of OpenMM) to rank analogues.

### ✏️ Exercise 1

Using `lig_rmsd`, `core_rmsd`, and the per-atom deviation from the crystal, find which **ligand atoms move most**
during the simulation. Print the five atoms with the largest average displacement.

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
disp = np.sqrt(((traj.xyz[:, lig_heavy] - ref.xyz[0, lig_heavy]) ** 2).sum(-1)).mean(0) * 10
names = [traj.topology.atom(i).name for i in lig_heavy]
print(pd.Series(disp, index=names).sort_values(ascending=False).head(5).round(2))
```

</details>

### ✏️ Exercise 2

The gatekeeper residue **Thr766** (UniProt T790; its mutation to methionine causes erlotinib resistance) lines the
back pocket. Compute the minimum distance between Thr766 side-chain heavy atoms and the erlotinib acetylene carbons
(`C1`, `C2`) over time.

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
thr = traj.topology.select("protein and resSeq 766 and sidechain and not element H")
acet = traj.topology.select("resname AQ4 and name C1 C2")
d = md.compute_distances(traj, np.array(list(product(acet, thr)))).min(1) * 10
print(f"Thr766 ... acetylene: mean {d.mean():.2f} Å, range {d.min():.2f}-{d.max():.2f} Å")
```

</details>

### ✏️ Exercise 3

Compare the per-residue Cα RMSF of the kinase during the simulation with the crystallographic B-factors (from
`xtal`'s PDB file, column 61–66). Which regions are most flexible in both? (Hint: `md.rmsf(traj, ref, atom_indices=ca)`.)

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
rmsf = md.rmsf(traj, ref, atom_indices=ca) * 10
resseq = [traj.topology.atom(i).residue.resSeq for i in ca]
bfac = {int(l[22:26]): float(l[60:66]) for l in open(f"{DATA_DIR}/structures/1M17.pdb") if l.startswith("ATOM") and l[12:16].strip() == "CA"}
dfr = pd.DataFrame({"resSeq": resseq, "RMSF": rmsf, "B": [bfac.get(r, np.nan) for r in resseq]})
print("correlation:", round(dfr[["RMSF", "B"]].corr().iloc[0, 1], 2))
print(dfr.sort_values("RMSF", ascending=False).head(8))
```

</details>

## Key takeaways

- Ligands need bond orders, hydrogens, a protonation decision, and general force-field parameters (GAFF2/OpenFF/CGenFF).
- openmmforcefields plugs GAFF2/AM1-BCC parameters straight into an OpenMM ForceField alongside amber14.
- Check pose stability with protein-aligned ligand RMSD and key interactions such as the kinase hinge H-bond.
- Contact frequencies and interaction fingerprints summarize how a drug binds over time.

**Next:** [12 · Good practices, pitfalls, and a mini project](12_good_practices_mini_project.ipynb).